# Data Transformation and Feature Engineering
**Unit 3 · Lesson 5 | PV Time Series**

**Load → encode → standardize → save.** Continue with the chronological CSVs from Lesson 2. Lesson 3 found no established need for cleaning, so we do not repeat EDA or change measurements.

Save two versions of each partition: encoded features in source units, and encoded features with standardized numerical measurements. The original CSVs remain unchanged. No JSON, reports, serialized objects or extra artifact folders are required.

## 1. Load the prepared partitions
Run locally from `chapter_03/`, or edit `PROJECT_ROOT`. Shared data lives in `data/processed/pv_time_series/`, outside chapter folders.

**Colab:** runtime storage is temporary. Keep the original Lesson 2 CSVs and generated outputs on Drive or download them before ending the session. Saving a notebook does not save runtime CSVs. Use a mounted Drive project root or the optional `gdown` example.

Run the imports, then choose **one** loading-location option below. Local setup is active by default; uncomment the alternative you need. Then run the shared CSV-reading cell.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
import gdown

### Option A — Local project

In [2]:
# PROJECT_ROOT = Path("..").resolve()
# DATA_DIR = PROJECT_ROOT / "data/processed/pv_time_series"

### Option B — Google Colab with Google Drive
Uncomment this cell to use persistent Drive storage.

In [3]:
# from google.colab import drive
# drive.mount("/content/drive")
# PROJECT_ROOT = Path("/content/drive/MyDrive/pv_course")
# DATA_DIR = PROJECT_ROOT / "data/processed/pv_time_series"

### Option C — Download prepared CSVs using gdown
Replace the file IDs and uncomment. These downloads use temporary Colab storage; preserve the outputs before ending the session.

In [4]:
DATA_DIR = Path("/content/data/processed/pv_time_series")
DATA_DIR.mkdir(parents=True, exist_ok=True)
# gdown.download(id="TRAIN_FILE_ID", output=str(DATA_DIR / "pv_time_series_train.csv"), quiet=False)
# gdown.download(id="VALIDATION_FILE_ID", output=str(DATA_DIR / "pv_time_series_validation.csv"), quiet=False)
# gdown.download(id="TEST_FILE_ID", output=str(DATA_DIR / "pv_time_series_test.csv"), quiet=False)
gdown.download(id="1nZuDEW1IYjKRMpuvPLDROXXDKaX3ip3R", output=str(DATA_DIR / "pv_time_series_train.csv"), quiet=False)
gdown.download(id="1GgaPyt9B9dslNzAva2BlNNLV5BJt2ti7", output=str(DATA_DIR / "pv_time_series_validation.csv"), quiet=False)
gdown.download(id="1MCLg7Dk3G7jrI_hNGz6mynE7v0Re1upx", output=str(DATA_DIR / "pv_time_series_test.csv"), quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1nZuDEW1IYjKRMpuvPLDROXXDKaX3ip3R
To: /content/data/processed/pv_time_series/pv_time_series_train.csv
100%|██████████| 623k/623k [00:00<00:00, 132MB/s]
Downloading...
From: https://drive.google.com/uc?id=1GgaPyt9B9dslNzAva2BlNNLV5BJt2ti7
To: /content/data/processed/pv_time_series/pv_time_series_validation.csv
100%|██████████| 139k/139k [00:00<00:00, 60.1MB/s]
Downloading...
From: https://drive.google.com/uc?id=1MCLg7Dk3G7jrI_hNGz6mynE7v0Re1upx
To: /content/data/processed/pv_time_series/pv_time_series_test.csv
100%|██████████| 132k/132k [00:00<00:00, 90.5MB/s]


'/content/data/processed/pv_time_series/pv_time_series_test.csv'

### Read the three CSVs
Run after choosing a data location.

In [5]:
train_data = pd.read_csv(DATA_DIR / "pv_time_series_train.csv")
validation_data = pd.read_csv(DATA_DIR / "pv_time_series_validation.csv")
test_data = pd.read_csv(DATA_DIR / "pv_time_series_test.csv")

print("Train:", train_data.shape)
print("Validation:", validation_data.shape)
print("Test:", test_data.shape)
display(train_data.head())

Train: (4956, 21)
Validation: (1062, 21)
Test: (1062, 21)


,day,cosine_of_day,hour,cosine_of_hour,air_temperature,dewpoint,wind_speed,air_pressure,weather_condition,solar_radiation,...,pv_1_power,pv_2_cosine_value_of_the_solar_incidence_angle,pv_2_power,pv_3_cosine_value_of_the_solar_incidence_angle,pv_3_power,pv_4_cosine_value_of_the_solar_incidence_angle,pv_4_power,pv_5_cosine_value_of_the_solar_incidence_angle,pv_5_power,total_pv_power
0,1,0.999852,1,1.0000,2,81,6,900.2,10,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0
1,1,0.999852,2,0.9659,-1,86,0,900.2,10,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0
2,1,0.999852,3,0.8660,0,87,6,900.2,10,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0
3,1,0.999852,4,0.7071,-1,86,2,900.2,10,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0
4,1,0.999852,5,0.5000,-1,86,6,900.2,10,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0


## 2. Ordinal encode the weather code
Keep weather as **one input column**, rather than expanding it into multiple indicator columns. `OrdinalEncoder` learns the category vocabulary from train only and assigns consecutive numeric codes. Apply the same mapping to validation/test.

For this training file, source codes `[1, 2, 3, 4, 5, 6, 7, 9, 10]` map to `[0, 1, 2, 3, 4, 5, 6, 7, 8]`. The table below shows the actual learned mapping. An unseen category receives `-1` and is reported; it is not added to the training vocabulary.

**Trade-off:** this compact representation is a project choice, not evidence that weather conditions have a natural order. A neural network consuming these numbers as a scalar sees numeric order and distances that may not reflect category meaning. Standardization does not remove that assumption. Evaluate its effect during modeling; ordinal encoding is not universally better than one-hot encoding.

We retain the existing cosine features and do not add extra hour sine/cosine columns without evaluating their need. No log transforms, interactions or automatic feature deletion are added just to cover slide topics.

In [6]:
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
encoder.fit(train_data[["weather_condition"]])
display(pd.DataFrame({
    "original_weather_code": encoder.categories_[0],
    "encoded_weather_code": range(len(encoder.categories_[0])),
}))

def encode_weather(data):
    unknown = ~data["weather_condition"].isin(encoder.categories_[0])
    print("Rows with unknown weather codes:", int(unknown.sum()))
    result = data.copy()
    result["weather_condition"] = encoder.transform(data[["weather_condition"]]).ravel()
    return result

train_encoded = encode_weather(train_data)
validation_encoded = encode_weather(validation_data)
test_encoded = encode_weather(test_data)
display(train_encoded[["weather_condition"]].head())

,original_weather_code,encoded_weather_code
0,1,0
1,2,1
2,3,2
3,4,3
4,5,4
5,6,5
6,7,6
7,9,7
8,10,8


Rows with unknown weather codes: 0
Rows with unknown weather codes: 0
Rows with unknown weather codes: 0


,weather_condition
0,8.0
1,8.0
2,8.0
3,8.0
4,8.0


## 3. Standardize numerical features
Use `StandardScaler`: subtract the training mean and divide by the training standard deviation. Standardization is not min–max normalization: results are not restricted to [0, 1], and later partitions need not have mean zero or standard deviation one.

- Fit on **train only**; use the same scaler for validation/test.
- Standardize numerical measurement and existing cosine columns.
- Keep `day` and `hour` unchanged as alignment keys, not automatically approved model inputs.
- Standardize the ordinal weather column too, since it is used as a scalar numerical input here. Its category mapping remains unchanged in the encoded CSVs; the standardized CSVs contain its scaled values. The unknown code `-1` is transformed by the same training-fitted scaler.

The standardized version supports scale-sensitive models such as neural networks and SVR. The encoded, unscaled version remains available for models that do not need standardization.

**Output timing:** `total_pv_power` is a current-interval measurement and is standardized here as a potential historical input. It is **not an aligned next-hour target**. Construct future targets from the original or unscaled CSV, not by accidentally shifting the standardized output column. Confirm measurement availability before selecting model inputs.

In [11]:
train_data["weather_condition"]

,weather_condition
0,10
1,10
2,10
3,10
4,10
...,...
4951,10
4952,10
4953,10
4954,10


In [10]:
train_data.columns

Index(['day', 'cosine_of_day', 'hour', 'cosine_of_hour', 'air_temperature',
       'dewpoint', 'wind_speed', 'air_pressure', 'weather_condition',
       'solar_radiation', 'pv_1_cosine_value_of_the_solar_incidence_angle',
       'pv_1_power', 'pv_2_cosine_value_of_the_solar_incidence_angle',
       'pv_2_power', 'pv_3_cosine_value_of_the_solar_incidence_angle',
       'pv_3_power', 'pv_4_cosine_value_of_the_solar_incidence_angle',
       'pv_4_power', 'pv_5_cosine_value_of_the_solar_incidence_angle',
       'pv_5_power', 'total_pv_power'],
      dtype='object')

In [7]:
numeric_columns = [
    column for column in train_data.columns
    if column not in ["day", "hour"]
]
scaler = StandardScaler()
scaler.fit(train_encoded[numeric_columns])

def standardize_features(data):
    result = data.copy()
    result[numeric_columns] = scaler.transform(data[numeric_columns])
    return result

train_standardized = standardize_features(train_encoded)
validation_standardized = standardize_features(validation_encoded)
test_standardized = standardize_features(test_encoded)
display(train_standardized.head())

,day,cosine_of_day,hour,cosine_of_hour,air_temperature,dewpoint,wind_speed,air_pressure,weather_condition,solar_radiation,...,pv_1_power,pv_2_cosine_value_of_the_solar_incidence_angle,pv_2_power,pv_3_cosine_value_of_the_solar_incidence_angle,pv_3_power,pv_4_cosine_value_of_the_solar_incidence_angle,pv_4_power,pv_5_cosine_value_of_the_solar_incidence_angle,pv_5_power,total_pv_power
0,1,1.524169,1,1.413955,-1.000664,1.167171,-0.905631,0.605426,0.891925,-0.713821,...,-0.720395,-0.703473,-0.671417,-0.739408,-0.679489,-0.767738,-0.698955,-0.789836,-0.712872,-0.706353
1,1,1.524169,2,1.365729,-1.300730,1.398667,-1.604835,0.605426,0.891925,-0.713821,...,-0.720395,-0.703473,-0.671417,-0.739408,-0.679489,-0.767738,-0.698955,-0.789836,-0.712872,-0.706353
2,1,1.524169,3,1.224447,-1.200708,1.444966,-0.905631,0.605426,0.891925,-0.713821,...,-0.720395,-0.703473,-0.671417,-0.739408,-0.679489,-0.767738,-0.698955,-0.789836,-0.712872,-0.706353
3,1,1.524169,4,0.999724,-1.300730,1.398667,-1.371767,0.605426,0.891925,-0.713821,...,-0.720395,-0.703473,-0.671417,-0.739408,-0.679489,-0.767738,-0.698955,-0.789836,-0.712872,-0.706353
4,1,1.524169,5,0.706835,-1.300730,1.398667,-0.905631,0.605426,0.891925,-0.713821,...,-0.720395,-0.703473,-0.671417,-0.739408,-0.679489,-0.767738,-0.698955,-0.789836,-0.712872,-0.706353


## 4. Save the CSVs
Save each table directly beside the prepared CSVs with a new name. Choose the encoded or standardized version later according to the model. Rerunning these cells refreshes only these output files, not the original split CSVs.

### Encoded observations — original numerical units

In [8]:
train_encoded.to_csv(DATA_DIR / "pv_time_series_train_encoded.csv", index=False)
validation_encoded.to_csv(DATA_DIR / "pv_time_series_validation_encoded.csv", index=False)
test_encoded.to_csv(DATA_DIR / "pv_time_series_test_encoded.csv", index=False)

### Standardized observations

In [9]:
train_standardized.to_csv(DATA_DIR / "pv_time_series_train_standardized.csv", index=False)
validation_standardized.to_csv(DATA_DIR / "pv_time_series_validation_standardized.csv", index=False)
test_standardized.to_csv(DATA_DIR / "pv_time_series_test_standardized.csv", index=False)

## 5. Handoff to modeling
- Use the **encoded** or **standardized** CSVs according to the model; do not standardize the latter a second time.
- Build next-hour targets from unscaled observations with time-key alignment, accounting for gaps and split boundaries. Keep `day` and `hour` until this alignment is complete, then choose actual model inputs explicitly. These exports are transformed observations, not completed `(X, y)` samples.
- No measurement rows were deleted and no future targets or arbitrary classes were created.
- Validation/test were transformed with training-fitted parameters, not used to choose or fit transformations.
- For temporal cross-validation, refit the encoder and scaler inside each training fold from the original inputs. Do not reuse globally transformed training CSVs across those folds.
- This notebook intentionally saves only CSVs. The fitted `encoder` and `scaler` remain in memory; rerun from the same original training CSV to reproduce them. A deployed model will need its fitted preprocessing saved with it at the modeling stage.
- In Colab, preserve these six output CSVs in persistent storage before ending the runtime.